# NonNULL Framework V1.0 — Official 1-Click Google Colab Pipeline

**Author**: Giang Nhan Nghia (*The Non-Null Project*)  
This notebook runs the complete **NonNULL Framework V1.0** pipeline end-to-end on Google Colab (recommended: **L4 24GB** for Qwen-3B/7B or **A100 80GB** for Llama-3.1-8B + HarmBench/MT-Bench):
1. GPU Check & Environment Setup
2. Unit Test Suite (`unittest`)
3. SFT + `EulerPhaseLoss` + `ContextPairedValueCritic` Training
4. Empathic DPO Alignment (Round 1)
5. 25-Scenario Genesis Evaluation + Automated Data Scar Extraction
6. Closed-Loop Scar Self-Healing (Round 2)
7. Out-of-Distribution (`OOD-20`) & Multi-Turn / Zero-Point ($Z$) General IQ Stress-Tests
8. Public Community Benchmarks (**HarmBench** + **MT-Bench**)
9. Double-Blind Evaluation Decoder & **Cohen's Kappa** (`score_blind_eval.py`)


## 1. GPU Check & Repository Setup


In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# Option A: Clone from GitHub (replace with your published repo URL)
!git clone https://github.com/GiangNhanNghia/NonNULL-Framework.git
%cd NonNULL-Framework

# Option B: Unzip uploaded NonNULL_Framework_V1.0_EN.zip on Colab
if os.path.exists("/content/NonNULL_Framework_V1.0_EN.zip"):
    !unzip -o -q /content/NonNULL_Framework_V1.0_EN.zip -d /content/
    %cd /content/NonNULL_Framework_V1.0_EN


In [ ]:
!pip install -q -r requirements.txt
!python3 -m unittest discover -s tests -v


## 2. Select Backbone Model (3B / 7B Qwen or 8B Llama)


In [ ]:
# Verified backbones:
# MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"                       # Minimum viable scale (L4 / T4)
# MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"           # Production Qwen scale (L4)
MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"      # Cross-architecture scale (A100)
TAG = "llama8b"
print("Selected:", MODEL_ID, "| Tag:", TAG)


## 3. SFT (`EulerPhaseLoss`) & `ContextPairedValueCritic` Training


In [ ]:
!python3 run_training.py \
  --model_id "$MODEL_ID" \
  --data_path ./nonnull_train_100.jsonl \
  --output_dir ./output/euler_checkpoints \
  --adapter_name "adapter_B_vol4_$TAG" \
  --epochs 2 --batch_size 2 --learning_rate 2e-4 --seed 42


## 4. Empathic DPO Alignment (Round 1)


In [ ]:
!python3 run_dpo_v1.py \
  --model_id "$MODEL_ID" \
  --init_adapter "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --output_adapter "./output/euler_checkpoints/adapter_B_dpo_$TAG" \
  --dpo_data ./dpo_train_50.jsonl \
  --val_data ./dpo_val_10.jsonl \
  --epochs 1 --lr 2e-5 --beta 0.1 --empathic_gamma 0.05 --sft_weight 0.1


## 5. Round 1 Evaluation (25 Scenarios) & Data Scar Extraction


In [ ]:
!python3 evaluate_genesis.py \
  --model_id "$MODEL_ID" \
  --adapter_b "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --adapter_c "./output/euler_checkpoints/adapter_B_dpo_$TAG" \
  --include_vol4 --use_consensus --num_samples 3 \
  --test_data ./nonnull_test_25.jsonl \
  --output "eval_${TAG}_r1.json" \
  --blind_csv "blind_${TAG}_r1.csv" \
  --blind_key "blind_${TAG}_r1_key.json" \
  --scars_out "scars_${TAG}.jsonl"


## 6. Closed-Loop Scar Self-Healing (Round 2) & Verification


In [ ]:
!python3 run_dpo_v1.py \
  --model_id "$MODEL_ID" \
  --init_adapter "./output/euler_checkpoints/adapter_B_dpo_$TAG" \
  --output_adapter "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --dpo_data ./dpo_train_50.jsonl \
  --val_data ./dpo_val_10.jsonl \
  --scars_file "scars_${TAG}.jsonl" \
  --epochs 1 --lr 2e-5 --beta 0.1 --empathic_gamma 0.05 --sft_weight 0.1

!python3 evaluate_genesis.py \
  --model_id "$MODEL_ID" \
  --adapter_b "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --adapter_c "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --include_vol4 --use_consensus --num_samples 3 \
  --test_data ./nonnull_test_25.jsonl \
  --output "eval_${TAG}_healed.json" \
  --blind_csv "blind_${TAG}_healed.csv" \
  --blind_key "blind_${TAG}_healed_key.json"


## 7. Out-of-Distribution (`OOD-20`) & Multi-Turn / Zero-Point ($Z$) IQ Stress-Tests


In [ ]:
!python3 evaluate_genesis.py \
  --model_id "$MODEL_ID" \
  --adapter_b "./output/euler_checkpoints/adapter_B_vol4_$TAG" \
  --adapter_c "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --include_vol4 --use_consensus --num_samples 3 \
  --test_data ./nonnull_ood_20.jsonl \
  --output "eval_${TAG}_ood20.json" \
  --blind_csv "blind_${TAG}_ood20.csv" \
  --blind_key "blind_${TAG}_ood20_key.json"

!python3 evaluate_extended.py \
  --model_id "$MODEL_ID" \
  --adapter_c "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --output "eval_${TAG}_extended.json"


## 8. Public Community Benchmarks: HarmBench & MT-Bench


In [ ]:
# Evaluates BASE vs NonNULL on 100 HarmBench behaviors (with official 13B classifier) & 80 MT-Bench questions
!python3 eval_public_benchmarks.py \
  --bench all \
  --model_id "$MODEL_ID" \
  --adapter_path "./output/euler_checkpoints/adapter_B_healed_$TAG" \
  --harmbench_subset standard \
  --harmbench_samples 100 \
  --run_harmbench_cls \
  --mtbench_samples 80 \
  --output_dir "./benchmark_outputs_${TAG}"


## 9. Double-Blind Evaluation Decoder & Cohen's Kappa


In [ ]:
# Run after two independent raters fill in blind_${TAG}_healed.csv
# !python3 score_blind_eval.py \
#   --csv "blind_${TAG}_healed.csv" \
#   --key "blind_${TAG}_healed_key.json" \
#   --out_md "blind_${TAG}_summary.md"
